# Stroke order filtering

Notebook helper to map one-character-per-line Chinese input to the existing stroke-order CSV, then derive the iframe embed HTML and image URL for each matched character.

In [11]:
from pathlib import Path
from typing import Iterable

import pandas as pd
from IPython.display import HTML, display

# CSV_PATH = Path(r"d:\repos\chinese-docs\docs\resources\List of Stroke Order Animation Embed URLs.csv")
CSV_PATH = Path("docs/resources/List of Stroke Order Animation Embed URLs.csv")
IMAGE_URL_TEMPLATE = "https://www.strokeorder.com/assets/bishun/guide/{id}.png"


def load_stroke_lookup(csv_path: Path = CSV_PATH) -> pd.DataFrame:
    df = pd.read_csv(csv_path, dtype=str)
    df.columns = ["no", "unicode_decimal", "character", "embed_html"]
    for column in df.columns:
        df[column] = df[column].astype(str).str.strip()
    df["image_url"] = df["unicode_decimal"].map(lambda value: IMAGE_URL_TEMPLATE.format(id=value))
    return df


def extract_characters(source: str | Path | Iterable[str]) -> list[str]:
    if isinstance(source, Path):
        text = source.read_text(encoding="utf-8")
    elif isinstance(source, str):
        if "\n" not in source and Path(source.strip()).exists():
            text = Path(source.strip()).read_text(encoding="utf-8")
        else:
            text = source
    else:
        text = "\n".join(str(item) for item in source)

    characters: list[str] = []
    seen: set[str] = set()
    for line in text.splitlines():
        character = line.strip()
        if len(character) != 1 or character in seen:
            continue
        seen.add(character)
        characters.append(character)
    return characters


def filter_stroke_entries(source: str | Path | Iterable[str], csv_path: Path = CSV_PATH) -> pd.DataFrame:
    characters = extract_characters(source)
    lookup = load_stroke_lookup(csv_path).set_index("character", drop=False)
    rows: list[dict[str, str]] = []

    for character in characters:
        if character not in lookup.index:
            continue
        record = lookup.loc[character]
        if isinstance(record, pd.DataFrame):
            record = record.iloc[0]
        rows.append(
            {
                "character": record["character"],
                "unicode_decimal": record["unicode_decimal"],
                "embed_html": record["embed_html"],
                "image_url": record["image_url"],
                "csv_no": record["no"],
            }
        )

    return pd.DataFrame(rows, columns=["character", "unicode_decimal", "embed_html", "image_url", "csv_no"])


def to_markdown_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "| character | unicode_decimal | embed_html | image_url | csv_no |\n| --- | --- | --- | --- | --- |"

    header = "| " + " | ".join(df.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(df.columns)) + " |"
    lines = [header, separator]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(row[column]) for column in df.columns) + " |")
    return "\n".join(lines)


def render_results(source: str | Path | Iterable[str], csv_path: Path = CSV_PATH) -> pd.DataFrame:
    result = filter_stroke_entries(source, csv_path=csv_path)
    display(result)
    display(HTML(result.to_html(index=False, escape=False)))
    print(to_markdown_table(result))
    return result

In [17]:
sample_input = """
是
学
生
工
程
师
"""

result = render_results(sample_input)
print("First matched row:")
print(result.iloc[0].to_dict())
print("Zero image URL:")
# print(result.loc[result["character"] == "零", "image_url"].iloc[0])

output_csv = Path("stroke_filtering_output.csv")
result.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"Wrote {output_csv.resolve()}")

,character,unicode_decimal,embed_html,image_url,csv_no
0,是,26159,<iframe src='https://stroke-order.learningweb....,https://www.strokeorder.com/assets/bishun/guid...,1367
1,学,23398,<iframe src='https://www.strokeorder.com/asset...,https://www.strokeorder.com/assets/bishun/guid...,23398
2,生,29983,<iframe src='https://stroke-order.learningweb....,https://www.strokeorder.com/assets/bishun/guid...,275
3,工,24037,<iframe src='https://stroke-order.learningweb....,https://www.strokeorder.com/assets/bishun/guid...,56
4,程,31243,<iframe src='https://stroke-order.learningweb....,https://www.strokeorder.com/assets/bishun/guid...,2979
5,师,24072,<iframe src='https://www.strokeorder.com/asset...,https://www.strokeorder.com/assets/bishun/guid...,24072


character,unicode_decimal,embed_html,image_url,csv_no
是,26159,,https://www.strokeorder.com/assets/bishun/guide/26159.png,1367
学,23398,,https://www.strokeorder.com/assets/bishun/guide/23398.png,23398
生,29983,,https://www.strokeorder.com/assets/bishun/guide/29983.png,275
工,24037,,https://www.strokeorder.com/assets/bishun/guide/24037.png,56
程,31243,,https://www.strokeorder.com/assets/bishun/guide/31243.png,2979
师,24072,,https://www.strokeorder.com/assets/bishun/guide/24072.png,24072


| character | unicode_decimal | embed_html | image_url | csv_no |
| --- | --- | --- | --- | --- |
| 是 | 26159 | <iframe src='https://stroke-order.learningweb.moe.edu.tw/dictFrame.jsp?ID=26159' frameborder=0 width=300 height=520 allow='fullscreen'></iframe> | https://www.strokeorder.com/assets/bishun/guide/26159.png | 1367 |
| 学 | 23398 | <iframe src='https://www.strokeorder.com/assets/bishun/animation/23398.gif' frameborder=0 width=300 height=520 allow='fullscreen'></iframe> | https://www.strokeorder.com/assets/bishun/guide/23398.png | 23398 |
| 生 | 29983 | <iframe src='https://stroke-order.learningweb.moe.edu.tw/dictFrame.jsp?ID=29983' frameborder=0 width=300 height=520 allow='fullscreen'></iframe> | https://www.strokeorder.com/assets/bishun/guide/29983.png | 275 |
| 工 | 24037 | <iframe src='https://stroke-order.learningweb.moe.edu.tw/dictFrame.jsp?ID=24037' frameborder=0 width=300 height=520 allow='fullscreen'></iframe> | https://www.strokeorder.com/assets/bishun/guide/24037.png | 56